# Seq2seq với attention Bahdanau và Luong

Bạn tự viết hai cơ chế attention kinh điển, ghép chúng vào một model encoder-decoder GRU, rồi đo attention giúp gì:
trước trên bài đảo ngược chuỗi (biết đáp án, thấy rõ nút thắt cổ chai), sau trên dịch Anh-Việt thật (IWSLT'15, câu
ngắn, vừa sức CPU). Dịch máy đầy đủ với BPE và beam search là bài Advanced 01.

**Cách làm bài**
- Cell chấm điểm chỉ định nghĩa hàm, lớp. Điền vào chỗ `# TODO`.
- Cell gắn tag `explore` (dòng đầu `# explore: ...`) đọc dữ liệu thật, train, vẽ; test bỏ qua chúng.
- Quy ước shape: batch trước (`batch_first`). `keys` là output encoder `(N, S, enc_dim)`, `mask` là `BoolTensor (N, S)`,
  `True` ở token thật, `False` ở padding.
- Chấm: `uv run pytest tracks/ai-engineer/modules/04-nlp-sequence-models/exercises/intermediate/03-seq2seq-attention -q`

In [ ]:
# explore: giới hạn luồng, tìm gốc repo, tải IWSLT'15 en-vi nếu chưa có (31 MB)
import os

os.environ.setdefault("OMP_NUM_THREADS", "4")   # đặt TRƯỚC khi import numpy/torch
import sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path.insert(0, str(ROOT))
from data.download_dataset import ensure

IWSLT = ensure("iwslt15-en-vi")
print(sorted(p.name for p in IWSLT.iterdir()))

In [ ]:
import html
import time
from collections import Counter

import numpy as np
import sacrebleu
import torch
import torch.nn as nn
import torch.nn.functional as F

PAD, UNK, BOS, EOS = 0, 1, 2, 3
SPECIALS = ["<pad>", "<unk>", "<bos>", "<eos>"]

## 1. Hai cơ chế attention

Cho query $$q$$ (state của decoder) và các key $$k_1, \dots, k_S$$ (output encoder ở từng vị trí nguồn):

$$
e_j = \text{score}(q, k_j), \qquad \alpha = \mathrm{softmax}(e) \text{ (chỉ trên vị trí thật)}, \qquad c = \sum_j \alpha_j k_j
$$

- `BahdanauAttention(enc_dim, dec_dim, attn_dim)` (additive): $$e_j = v^\top \tanh(W_q q + W_k k_j)$$. Ba lớp
  `nn.Linear` **không bias**, đặt tên đúng `W_q` (dec_dim → attn_dim), `W_k` (enc_dim → attn_dim), `v` (attn_dim → 1).
- `LuongAttention(enc_dim, dec_dim, kind)` (multiplicative): `kind="dot"`: $$e_j = q^\top k_j$$ (cần
  `enc_dim == dec_dim`, khác thì `raise ValueError`); `kind="general"`: $$e_j = q^\top W k_j$$ với `W` là
  `nn.Linear(enc_dim, dec_dim, bias=False)`.
- Cả hai: `forward(query (N, dec_dim), keys (N, S, enc_dim), mask (N, S)) -> (context (N, enc_dim), weights (N, S))`.
  Vị trí padding có trọng số **đúng bằng 0** (gán score `-inf` trước softmax, không nhân mask sau softmax).

In [ ]:
class BahdanauAttention(nn.Module):
    def __init__(self, enc_dim: int, dec_dim: int, attn_dim: int):
        super().__init__()
        self.W_q = nn.Linear(dec_dim, attn_dim, bias=False)
        self.W_k = nn.Linear(enc_dim, attn_dim, bias=False)
        self.v = nn.Linear(attn_dim, 1, bias=False)

    def forward(self, query, keys, mask):
        scores = self.v(torch.tanh(self.W_q(query).unsqueeze(1) + self.W_k(keys))).squeeze(-1)   # (N, S)
        weights = torch.softmax(scores.masked_fill(~mask, float("-inf")), dim=-1)
        return torch.bmm(weights.unsqueeze(1), keys).squeeze(1), weights


class LuongAttention(nn.Module):
    def __init__(self, enc_dim: int, dec_dim: int, kind: str = "general"):
        super().__init__()
        self.kind = kind
        if kind == "general":
            self.W = nn.Linear(enc_dim, dec_dim, bias=False)
        elif kind == "dot":
            if enc_dim != dec_dim:
                raise ValueError("Luong dot cần enc_dim == dec_dim")
        else:
            raise ValueError(f"kind không hợp lệ: {kind}")

    def forward(self, query, keys, mask):
        k = self.W(keys) if self.kind == "general" else keys                                  # (N, S, dec_dim)
        scores = torch.bmm(k, query.unsqueeze(-1)).squeeze(-1)                                  # (N, S)
        weights = torch.softmax(scores.masked_fill(~mask, float("-inf")), dim=-1)
        return torch.bmm(weights.unsqueeze(1), keys).squeeze(1), weights

## 2. Encoder-decoder GRU

`Seq2Seq(src_vocab, tgt_vocab, emb, hidden, attn, dropout=0.0)`, `attn` là `"none"`, `"bahdanau"` hoặc `"luong"`:

- **Encoder**: `nn.Embedding(src_vocab, emb, padding_idx=0)` → dropout → GRU **hai chiều** `hidden` mỗi chiều,
  `batch_first`, chạy trên `pack_padded_sequence`. `keys` là output mọi bước `(N, S, 2*hidden)` (pad về đúng `S` cột của
  `src`). State đầu của decoder: $$s_0 = \tanh(W_b [h_{fwd}; h_{bwd}])$$ với `self.bridge = nn.Linear(2*hidden, hidden)` (đúng
  tên này), $$h_{fwd}$$ là state chiều xuôi sau token thật cuối, $$h_{bwd}$$ là state chiều ngược sau khi đọc về token đầu.
- `encode(src, src_len) -> (keys, mask, s0)`, `mask = src != PAD`.
- `step(y_prev, s, keys, mask) -> (logits, s_new, weights)`: một bước decoder, `y_prev` là `LongTensor (N,)`, `weights`
  là `None` khi không có attention. Thứ tự tính khác nhau theo kiểu, đúng như hai paper:

| `attn` | Một bước decoder |
| --- | --- |
| `"none"` | $$s_t = \mathrm{GRU}(e_t, s_{t-1})$$; logits $$= W_o s_t$$ |
| `"bahdanau"` | $$c_t = \mathrm{attn}(s_{t-1}, K)$$ (dùng state **cũ**); $$s_t = \mathrm{GRU}([e_t; c_t], s_{t-1})$$; logits $$= W_o [s_t; c_t; e_t]$$ |
| `"luong"` (general) | $$s_t = \mathrm{GRU}(e_t, s_{t-1})$$; $$c_t = \mathrm{attn}(s_t, K)$$ (dùng state **mới**); $$\tilde h_t = \tanh(W_c [s_t; c_t])$$; logits $$= W_o \tilde h_t$$ |

  với $$e_t$$ là embedding (qua dropout) của `y_prev`. Bahdanau: `attn_dim = hidden`.
- `forward(src, src_len, tgt_in) -> logits (N, T, tgt_vocab)`: teacher forcing, `tgt_in` bắt đầu bằng `<bos>`. Với
  `"none"` và `"luong"`, decoder GRU không phụ thuộc context, nên **chạy cả chuỗi `tgt_in` một lần** bằng `nn.GRU` rồi
  tính attention cho mọi bước bằng một phép `bmm` (nhanh hơn nhiều so với vòng lặp). Bahdanau buộc phải lặp từng bước.
  Kết quả (ở chế độ `eval`) phải **bằng** việc gọi `step` lần lượt; test kiểm điều này.

In [ ]:
class Seq2Seq(nn.Module):
    def __init__(self, src_vocab: int, tgt_vocab: int, emb: int, hidden: int, attn: str, dropout: float = 0.0):
        super().__init__()
        if attn not in ("none", "bahdanau", "luong"):
            raise ValueError(attn)
        self.attn_kind = attn
        H = hidden
        self.src_emb = nn.Embedding(src_vocab, emb, padding_idx=PAD)
        self.tgt_emb = nn.Embedding(tgt_vocab, emb, padding_idx=PAD)
        self.encoder = nn.GRU(emb, H, batch_first=True, bidirectional=True)
        self.bridge = nn.Linear(2 * H, H)
        self.drop = nn.Dropout(dropout)
        if attn == "bahdanau":
            self.attn = BahdanauAttention(2 * H, H, H)
            self.decoder = nn.GRU(emb + 2 * H, H, batch_first=True)
            self.out = nn.Linear(H + 2 * H + emb, tgt_vocab)
        elif attn == "luong":
            self.attn = LuongAttention(2 * H, H, "general")
            self.decoder = nn.GRU(emb, H, batch_first=True)
            self.combine = nn.Linear(H + 2 * H, H)
            self.out = nn.Linear(H, tgt_vocab)
        else:
            self.decoder = nn.GRU(emb, H, batch_first=True)
            self.out = nn.Linear(H, tgt_vocab)

    def encode(self, src, src_len):
        x = self.drop(self.src_emb(src))
        packed = nn.utils.rnn.pack_padded_sequence(x, src_len.cpu(), batch_first=True, enforce_sorted=False)
        out, h = self.encoder(packed)
        keys, _ = nn.utils.rnn.pad_packed_sequence(out, batch_first=True, total_length=src.shape[1])
        s0 = torch.tanh(self.bridge(torch.cat([h[0], h[1]], dim=-1)))
        return keys, src != PAD, s0

    def step(self, y_prev, s, keys, mask):
        e = self.drop(self.tgt_emb(y_prev))
        if self.attn_kind == "bahdanau":
            c, w = self.attn(s, keys, mask)
            _, h = self.decoder(torch.cat([e, c], dim=-1).unsqueeze(1), s.unsqueeze(0))
            s = h[0]
            return self.out(self.drop(torch.cat([s, c, e], dim=-1))), s, w
        _, h = self.decoder(e.unsqueeze(1), s.unsqueeze(0))
        s = h[0]
        if self.attn_kind == "luong":
            c, w = self.attn(s, keys, mask)
            return self.out(self.drop(torch.tanh(self.combine(torch.cat([s, c], dim=-1))))), s, w
        return self.out(self.drop(s)), s, None

    def forward(self, src, src_len, tgt_in):
        keys, mask, s = self.encode(src, src_len)
        if self.attn_kind == "bahdanau":
            logits = []
            for t in range(tgt_in.shape[1]):
                out, s, _ = self.step(tgt_in[:, t], s, keys, mask)
                logits.append(out)
            return torch.stack(logits, dim=1)
        S, _ = self.decoder(self.drop(self.tgt_emb(tgt_in)), s.unsqueeze(0))          # (N, T, H) một lần
        if self.attn_kind == "none":
            return self.out(self.drop(S))
        scores = torch.bmm(S, self.attn.W(keys).transpose(1, 2))                       # (N, T, S_src)
        weights = torch.softmax(scores.masked_fill(~mask.unsqueeze(1), float("-inf")), dim=-1)
        C = torch.bmm(weights, keys)                                                    # (N, T, 2H)
        return self.out(self.drop(torch.tanh(self.combine(torch.cat([S, C], dim=-1)))))

## 3. Decode và BLEU

- `greedy_decode(model, src, src_len, max_len) -> list[list[int]]`: ở chế độ `eval`, không gradient; bắt đầu bằng
  `<bos>`, mỗi bước lấy argmax, dừng khi **mọi** câu đã sinh `<eos>` hoặc đủ `max_len` bước. Trả token của từng câu, cắt
  trước `<eos>` đầu tiên (không gồm `<bos>`, `<eos>`).
- `corpus_bleu(hyps, refs) -> float`: BLEU của sacrebleu trên chuỗi đã tách từ sẵn: `tokenize="none"`, một reference
  mỗi câu.

In [ ]:
@torch.no_grad()
def greedy_decode(model: Seq2Seq, src, src_len, max_len: int) -> list[list[int]]:
    model.eval()
    keys, mask, s = model.encode(src, src_len)
    y = torch.full((src.shape[0],), BOS, dtype=torch.long)
    done = torch.zeros(src.shape[0], dtype=torch.bool)
    steps = []
    for _ in range(max_len):
        logits, s, _ = model.step(y, s, keys, mask)
        y = logits.argmax(-1)
        steps.append(y)
        done |= y == EOS
        if done.all():
            break
    out = []
    for row in torch.stack(steps, dim=1).tolist():
        out.append(row[:row.index(EOS)] if EOS in row else row)
    return out


def corpus_bleu(hyps: list[str], refs: list[str]) -> float:
    return sacrebleu.corpus_bleu(hyps, [refs], tokenize="none").score

## 4. Thí nghiệm 1: đảo ngược chuỗi

Input là chuỗi ký hiệu ngẫu nhiên dài 5 tới `L`, output là chính chuỗi đó viết ngược. Không có attention, encoder phải nén
cả chuỗi vào một vector $$s_0$$ cỡ `hidden`; có attention, decoder nhìn lại được từng vị trí. Hàm `pad` và
`reverse_batch` dưới đây cho sẵn (không chấm).

In [ ]:
def pad(seqs: list[list[int]]) -> tuple[torch.Tensor, torch.Tensor]:
    lengths = torch.tensor([len(s) for s in seqs])
    out = torch.full((len(seqs), int(lengths.max())), PAD, dtype=torch.long)
    for i, s in enumerate(seqs):
        out[i, :len(s)] = torch.tensor(s)
    return out, lengths


def reverse_batch(n: int, max_len: int, n_symbols: int, g: torch.Generator):
    seqs = [torch.randint(4, 4 + n_symbols, (int(torch.randint(5, max_len + 1, (1,), generator=g)),), generator=g).tolist()
            for _ in range(n)]
    src, src_len = pad(seqs)
    tgt, _ = pad([[BOS] + s[::-1] + [EOS] for s in seqs])
    return src, src_len, tgt

In [ ]:
# explore: none / luong / bahdanau trên bài đảo chuỗi dài tới 20 (hidden 32 cố ý nhỏ), 1 500 bước mỗi model
torch.set_num_threads(4)


def train_reverse(attn: str, max_len: int, steps: int = 1500, seed: int = 0):
    torch.manual_seed(seed)
    g = torch.Generator().manual_seed(seed)
    model = Seq2Seq(4 + 10, 4 + 10, emb=16, hidden=32, attn=attn)
    opt = torch.optim.Adam(model.parameters(), 3e-3)
    for _ in range(steps):
        src, src_len, tgt = reverse_batch(64, max_len, 10, g)
        logits = model(src, src_len, tgt[:, :-1])
        loss = F.cross_entropy(logits.reshape(-1, logits.shape[-1]), tgt[:, 1:].reshape(-1), ignore_index=PAD)
        opt.zero_grad()
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step()
    src, src_len, tgt = reverse_batch(500, max_len, 10, g)
    pred = greedy_decode(model, src, src_len, max_len + 2)
    gold = [t[1:t.index(EOS)] for t in tgt.tolist()]
    return model, float(np.mean([p == q for p, q in zip(pred, gold)]))


reverse_models = {}
for attn in ["none", "luong", "bahdanau"]:
    start = time.perf_counter()
    reverse_models[attn], acc = train_reverse(attn, max_len=20)
    print(f"{attn:9} đúng nguyên chuỗi: {acc:.3f} ({time.perf_counter() - start:.0f} s)")

In [ ]:
# explore: ma trận attention của model luong trên một chuỗi: đường chéo ngược
import matplotlib.pyplot as plt

model = reverse_models["luong"].eval()
src, src_len = pad([[4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 4, 5]])
with torch.no_grad():
    keys, mask, s = model.encode(src, src_len)
    y, rows = torch.tensor([BOS]), []
    for _ in range(12):
        logits, s, w = model.step(y, s, keys, mask)
        y = logits.argmax(-1)
        rows.append(w[0].numpy())
plt.imshow(np.array(rows), cmap="viridis")
plt.xlabel("vị trí nguồn"); plt.ylabel("bước sinh")
plt.title("Attention Luong trên bài đảo chuỗi")
plt.show()

## 5. Thí nghiệm 2: dịch Anh-Việt (IWSLT'15, câu ngắn)

Dữ liệu đã tách từ sẵn (dấu câu tách khỏi từ, ký tự đặc biệt dạng `&apos;`).

- `read_parallel(root, split, max_len) -> list[tuple[list[str], list[str]]]`: đọc `root / f"{split}.en"` và
  `root / f"{split}.vi"` (UTF-8), mỗi dòng `html.unescape`, `lower()`, tách theo khoảng trắng; nếu `max_len` không phải
  `None` thì chỉ giữ cặp mà **cả hai** phía có từ 1 tới `max_len` token.
- `build_vocab(seqs, size) -> list[str]`: `SPECIALS` rồi `size - 4` token nhiều nhất, hoà thì theo chữ cái.
- `encode(tokens, stoi) -> list[int]`: token lạ thành `<unk>`.

In [ ]:
def read_parallel(root, split: str, max_len: int | None) -> list[tuple[list[str], list[str]]]:
    def lines(lang: str) -> list[list[str]]:
        text = (root / f"{split}.{lang}").read_text(encoding="utf-8")
        return [html.unescape(line).lower().split() for line in text.splitlines()]

    pairs = list(zip(lines("en"), lines("vi")))
    if max_len is not None:
        pairs = [(s, t) for s, t in pairs if 1 <= len(s) <= max_len and 1 <= len(t) <= max_len]
    return pairs


def build_vocab(seqs: list[list[str]], size: int) -> list[str]:
    freq = Counter(w for s in seqs for w in s)
    return SPECIALS + sorted(freq, key=lambda w: (-freq[w], w))[:size - len(SPECIALS)]


def encode(tokens: list[str], stoi: dict[str, int]) -> list[int]:
    return [stoi.get(w, UNK) for w in tokens]

In [ ]:
# explore: 46 311 cặp câu ≤ 15 từ, vocab 4 000 mỗi phía
pairs = read_parallel(IWSLT, "train", max_len=15)
src_itos = build_vocab([s for s, _ in pairs], 4000)
tgt_itos = build_vocab([t for _, t in pairs], 4000)
src_stoi = {w: i for i, w in enumerate(src_itos)}
tgt_stoi = {w: i for i, w in enumerate(tgt_itos)}
S = [encode(s, src_stoi) for s, _ in pairs]
T = [[BOS] + encode(t, tgt_stoi) + [EOS] for _, t in pairs]
unk = sum(x == UNK for t in T for x in t) / sum(map(len, T))
print(len(pairs), "cặp | tỉ lệ <unk> phía tiếng Việt:", round(unk, 4))
test = read_parallel(IWSLT, "tst2013", max_len=15)
print(len(test), "câu test ≤ 15 từ")

In [ ]:
# explore: train none và luong, 3 epoch mỗi model (khoảng 1-2 phút mỗi epoch trên CPU 4 luồng)
def train_translation(attn: str, epochs: int = 3, seed: int = 0):
    torch.manual_seed(seed)
    g = torch.Generator().manual_seed(seed)
    model = Seq2Seq(len(src_itos), len(tgt_itos), emb=64, hidden=128, attn=attn, dropout=0.2)
    opt = torch.optim.Adam(model.parameters(), 1e-3)
    for epoch in range(epochs):
        model.train()
        perm, total, n_tok, start = torch.randperm(len(S), generator=g).tolist(), 0.0, 0, time.perf_counter()
        for i in range(0, len(S), 64):
            b = perm[i:i + 64]
            src, src_len = pad([S[j] for j in b])
            tgt, _ = pad([T[j] for j in b])
            logits = model(src, src_len, tgt[:, :-1])
            loss = F.cross_entropy(logits.reshape(-1, logits.shape[-1]), tgt[:, 1:].reshape(-1), ignore_index=PAD)
            opt.zero_grad()
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
            n = int((tgt[:, 1:] != PAD).sum())
            total, n_tok = total + loss.item() * n, n_tok + n
        print(f"  {attn} epoch {epoch + 1}: train loss {total / n_tok:.3f}, perplexity {np.exp(total / n_tok):.1f},"
              f" {time.perf_counter() - start:.0f} s")
    return model


def translate(model, sentences: list[list[str]]) -> list[str]:
    hyps = []
    for i in range(0, len(sentences), 100):
        src, src_len = pad([encode(s, src_stoi) for s in sentences[i:i + 100]])
        hyps += [" ".join(tgt_itos[t] for t in out) for out in greedy_decode(model, src, src_len, 40)]
    return hyps


mt = {}
for attn in ["none", "luong"]:
    mt[attn] = train_translation(attn)
    hyps = translate(mt[attn], [s for s, _ in test])
    print(f"{attn}: BLEU tst2013 (câu ≤ 15 từ) = {corpus_bleu(hyps, [' '.join(t) for _, t in test]):.2f}")

In [ ]:
# explore: vài câu dịch và attention của một câu
hyps = {a: translate(m, [s for s, _ in test[:8]]) for a, m in mt.items()}
for k, (s, t) in enumerate(test[:8]):
    print("EN  :", " ".join(s)); print("REF :", " ".join(t))
    for a in mt:
        print(f"{a:5}:", hyps[a][k])
    print()

model = mt["luong"].eval()
sent = "i was very proud of my country .".split()
src, src_len = pad([encode(sent, src_stoi)])
with torch.no_grad():
    keys, mask, s = model.encode(src, src_len)
    y, rows, words = torch.tensor([BOS]), [], []
    for _ in range(15):
        logits, s, w = model.step(y, s, keys, mask)
        y = logits.argmax(-1)
        if y.item() == EOS:
            break
        rows.append(w[0].numpy()); words.append(tgt_itos[y.item()])
plt.figure(figsize=(6, 5))
plt.imshow(np.array(rows), cmap="viridis")
plt.xticks(range(len(sent)), sent, rotation=60); plt.yticks(range(len(words)), words)
plt.title("Attention Luong: en → vi")
plt.show()

## 6. Đọc kết quả

So với mục 6 phần lý thuyết. Trên bài đảo chuỗi, model không attention có `hidden` 32 không nén nổi chuỗi dài 20 vào một
vector, còn hai model attention gần như hoàn hảo và ma trận attention là một đường chéo ngược: nút thắt cổ chai là có thật.
Trên dịch thật, model nhỏ train 3 epoch trên CPU chỉ được BLEU một chữ số, attention hơn chưa tới 1 điểm. Nhìn heatmap của
câu "i was very proud of my country ." để biết vì sao: ở lần chạy của lời giải, mọi bước sinh đều dồn attention vào dấu
chấm cuối câu, tức model dùng attention như một vector tóm tắt cố định, chẳng khác gì không có attention. Attention cần
nhiều dữ liệu và thời gian train hơn mới học được căn chỉnh từ với từ (bài Advanced 01, GPU).

Sau khi chạy hết notebook, làm phần **trắc nghiệm** của bài trong app.